## Bonus Exercise: Bikeshare

Here is your chance to apply what you have learned to some real data. We will be using a bike sharing dataset, but a very different one from last class

Here you’re going to be working with publicly available bike data from the Bay Area Bike Share portal, specifically analyzing the 2017 year of data. First we will download the data and read it into spark, we will also rename some of the columns to meet the requirements of graphframes.

In [1]:
#Checking the installed Java version
!java -version
!pip install "pyspark==3.5.0" 
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

!java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)
Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://archive.ubuntu.com/ubuntu noble InRelease                        
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:7 https://archive.ubuntu.com/ubuntu noble-updates InRelease                
Hit:8 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 http://deb.wakemeops.com/wakemeops stable InRelease                      
Hit:10 https://archive.ubuntu.com/ubuntu noble-backports InRelease             
Hit:11 https://cloud.archive.ubuntu.com

In [2]:
%pip install graphframes-py==0.10.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GraphFramesWithSpark4") \
    .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.12:0.10.0") \
    .getOrCreate()

print(f"spark version: {spark.version}")
print("spark session created with graphframes package specified!")

:: loading settings :: url = jar:file:/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/zeus/.ivy2/cache
The jars for the packages stored in: /home/zeus/.ivy2/jars
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f2c6eb13-bc57-4e26-9342-3ef1cb5661ad;1.0
	confs: [default]
	found io.graphframes#graphframes-spark3_2.12;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 in central
:: resolution report :: resolve 223ms :: artifacts dl 8ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 from central in [default]
	io.graphframes#graphframes-spark3_2.12;0.10.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||  

spark version: 3.5.0
spark session created with graphframes package specified!


In [4]:
#import the package we just installed
from graphframes import *
#import data types - All data types of Spark SQL are located in the package of pyspark.sql.types
from pyspark.sql.types import *
#row can be used to create a row object by using named arguments
from pyspark.sql import Row
from pyspark.sql.functions import col


In [5]:
!wget -O /teamspace/studios/this_studio/week12/bikes/201508_trip_data.csv https://raw.githubusercontent.com/udacity/data-analyst/master/projects/bike_sharing/201508_trip_data.csv

--2025-11-26 17:58:55--  https://raw.githubusercontent.com/udacity/data-analyst/master/projects/bike_sharing/201508_trip_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 43012650 (41M) [text/plain]
Saving to: ‘/teamspace/studios/this_studio/week12/bikes/201508_trip_data.csv’

/teamspace/studios/ 100%[===================>]  41.02M   136MB/s    in 0.3s    

2025-11-26 17:58:56 (136 MB/s) - ‘/teamspace/studios/this_studio/week12/bikes/201508_trip_data.csv’ saved [43012650/43012650]



In [6]:
!wget -O /teamspace/studios/this_studio/week12/bikes/201508_station_data.csv https://raw.githubusercontent.com/udacity/data-analyst/master/projects/bike_sharing/201508_station_data.csv

--2025-11-26 17:58:56--  https://raw.githubusercontent.com/udacity/data-analyst/master/projects/bike_sharing/201508_station_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5272 (5.1K) [text/plain]
Saving to: ‘/teamspace/studios/this_studio/week12/bikes/201508_station_data.csv’

/teamspace/studios/ 100%[===================>]   5.15K  --.-KB/s    in 0s      

2025-11-26 17:58:56 (26.8 MB/s) - ‘/teamspace/studios/this_studio/week12/bikes/201508_station_data.csv’ saved [5272/5272]



In [7]:
tripDF = spark.read.csv('/teamspace/studios/this_studio/week12/bikes/201508_trip_data.csv',header=True, inferSchema = True)
tripDF.printSchema()

root
 |-- Trip ID: integer (nullable = true)
 |-- Duration: integer (nullable = true)
 |-- Start Date: string (nullable = true)
 |-- Start Station: string (nullable = true)
 |-- Start Terminal: integer (nullable = true)
 |-- End Date: string (nullable = true)
 |-- End Station: string (nullable = true)
 |-- End Terminal: integer (nullable = true)
 |-- Bike #: integer (nullable = true)
 |-- Subscriber Type: string (nullable = true)
 |-- Zip Code: string (nullable = true)



In [8]:
stationDF = spark.read.csv('/teamspace/studios/this_studio/week12/bikes/201508_station_data.csv',header=True, inferSchema = True)
stationDF.printSchema()

root
 |-- station_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dockcount: integer (nullable = true)
 |-- landmark: string (nullable = true)
 |-- installation: string (nullable = true)



In [9]:
stationVertices = stationDF.withColumnRenamed("name", "id").distinct()
tripEdges = tripDF.withColumnRenamed("Start Station", "src").withColumnRenamed("End Station", "dst")

In [10]:
tripEdges.show(10, False)

+-------+--------+---------------+---------------------------------------------+--------------+---------------+---------------------------------------------+------------+------+---------------+--------+
|Trip ID|Duration|Start Date     |src                                          |Start Terminal|End Date       |dst                                          |End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+---------------------------------------------+--------------+---------------+---------------------------------------------+------------+------+---------------+--------+
|913460 |765     |8/31/2015 23:26|Harry Bridges Plaza (Ferry Building)         |50            |8/31/2015 23:39|San Francisco Caltrain (Townsend at 4th)     |70          |288   |Subscriber     |2139    |
|913459 |1036    |8/31/2015 23:11|San Antonio Shopping Center                  |31            |8/31/2015 23:28|Mountain View City Hall                      |27          |35    |Subscriber 

In [11]:
stationVertices.show(10, False)

+----------+-------------------------------------+---------+-----------+---------+-------------+------------+
|station_id|id                                   |lat      |long       |dockcount|landmark     |installation|
+----------+-------------------------------------+---------+-----------+---------+-------------+------------+
|46        |Washington at Kearney                |37.795425|-122.404767|15       |San Francisco|8/19/2013   |
|41        |Clay at Battery                      |37.795001|-122.39997 |15       |San Francisco|8/19/2013   |
|33        |Rengstorff Avenue / California Street|37.400241|-122.099076|15       |Mountain View|8/16/2013   |
|29        |San Antonio Caltrain Station         |37.40694 |-122.106758|23       |Mountain View|8/15/2013   |
|42        |Davis at Jackson                     |37.79728 |-122.398436|15       |San Francisco|8/19/2013   |
|13        |St James Park                        |37.339301|-121.889937|15       |San Jose     |8/6/2013    |
|54       

Alright you now have the data formatted as you need to perform analysis. Take a look to make sure you understand what the data is. Basically you have bike stations (verticies) and trips between the stations (edges). 

Hint: in the cell above I used 'False' in the `show()`, this will be useful for you to prevent truncating station names. The highest ranked station should be San Jose Diridon Caltrain Station.

#####Create a graphframe and perform a pagerank to find the most import stations (make sure to set the maxIterations to 5).

In [12]:
#take the vertices and edges DataFrames and returns a GraphFrames object.
gbs = GraphFrame(stationVertices, tripEdges)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


In [13]:
#show the edges
gbs.edges.show(10, False)

+-------+--------+---------------+---------------------------------------------+--------------+---------------+---------------------------------------------+------------+------+---------------+--------+
|Trip ID|Duration|Start Date     |src                                          |Start Terminal|End Date       |dst                                          |End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+---------------------------------------------+--------------+---------------+---------------------------------------------+------------+------+---------------+--------+
|913460 |765     |8/31/2015 23:26|Harry Bridges Plaza (Ferry Building)         |50            |8/31/2015 23:39|San Francisco Caltrain (Townsend at 4th)     |70          |288   |Subscriber     |2139    |
|913459 |1036    |8/31/2015 23:11|San Antonio Shopping Center                  |31            |8/31/2015 23:28|Mountain View City Hall                      |27          |35    |Subscriber 

In [14]:
#Run the pageRank()
pageRanks = gbs.pageRank(resetProbability=0.15, maxIter = 5)

25/11/26 17:59:13 WARN BlockManager: Block rdd_114_0 already exists on this machine; not re-adding it
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [15]:
#Show() the vertices, ranked
pageRanks.vertices.orderBy(pageRanks.vertices.pagerank,ascending=False).show(10,False)

+----------+----------------------------------------+---------+-----------+---------+-------------+------------+------------------+
|station_id|id                                      |lat      |long       |dockcount|landmark     |installation|pagerank          |
+----------+----------------------------------------+---------+-----------+---------+-------------+------------+------------------+
|2         |San Jose Diridon Caltrain Station       |37.329732|-121.901782|27       |San Jose     |8/6/2013    |4.084086216116666 |
|70        |San Francisco Caltrain (Townsend at 4th)|37.776617|-122.39526 |19       |San Francisco|8/23/2013   |3.351543505799    |
|28        |Mountain View Caltrain Station          |37.394358|-122.076713|23       |Mountain View|8/15/2013   |2.5531183688195247|
|22        |Redwood City Caltrain Station           |37.486078|-122.232089|25       |Redwood City |8/15/2013   |2.4734490963466995|
|69        |San Francisco Caltrain 2 (330 Townsend) |37.7766  |-122.39547 |2

One question is: what are the most common trip paths? You can do this by performing a grouping operator and adding the edge counts together, think of how we used sparkSQL to implement SQL like queries last week. Remember the list of edges is just a dataframe so you can use SQL queries to find a result like this. In case you have forgotten this link has some information on groupBy:

https://sparkbyexamples.com/spark/using-groupby-on-dataframe/

The most common trip should be:

San Francisco Caltrain 2 (330 Townsend)  ->    Townsend at 7th.

In [17]:
#Create a gbsUnique graphframe
gbsUnique = GraphFrame(stationVertices, tripEdgesSum)

Remember that in this instance you’ve got a directed graph. That means that your trips are directional - from one location to another. Therefore you get access to a wealth of analysis that you can use. You can find the number of trips that go into a specific station and leave from a specific station.

One interesting question you could ask is what is the station with the highest ratio of in degrees to out degrees. As in, what station acts as a pure trip sink. A station where trips end at but rarely start from. This station will end up with a lot of excess bikes that will need to be re-distributed.

Consider the station with the highest ratio of intrips to outtrips, find all stations connected by 1 or 2 trips to this station using motifs.

## Some More Exercises

In [26]:
import pyspark.sql.functions as F

### Exercise 1 – Transfer hubs in the bike network

In this exercise, you will identify the key “transfer hubs” in the bike-share network. These are stations that are structurally important in the graph and also handle a large volume of trips. Use graph measures of centrality together with trip-level usage information to build a composite “hub” score, then find the top 5 stations according to this score. Interpret what makes these stations important in the system and whether they correspond to your intuition about major transfer points.


### Exercise 2 – Community structure and internal versus external flows

In this exercise, you will detect communities of stations in the bike network and use these communities to study travel patterns. First, identify groups of stations that form communities in the graph. Then, use the trip data to distinguish between “internal” trips that stay within the same community and “external” trips that cross from one community to another. Compare communities that are mostly self-contained with those that act as gateways, and discuss what this reveals about the organisation of mobility in the system.


### Exercise 3 – The giant component and peripheral sub-networks

In this exercise, you will explore global connectivity in the bike network. Start by finding the largest connected component, often called the “giant component,” and separate it from the smaller components. Compare these two sets of stations in terms of their number of docks, their coverage in the network, and their trip activity. Use your results to describe how much of the system is integrated into a single large network and what characterises the more peripheral or isolated sub-networks.


### Exercise 4 – Sources, sinks, and commuting patterns

In this exercise, you will characterise stations by how unbalanced their flows are. Some stations behave like “sources,” where many trips start but relatively few end, while others behave like “sinks,” receiving more trips than they send. Using the directed bike network, quantify net inflow and outflow for each station and use this to classify their commuting role. Relate these roles to station attributes such as landmark (city), installation date, or dock count, and discuss whether the main sources and sinks correspond to residential areas, business districts, or major transit nodes.


### Exercise 5 – Network robustness under hub failures

In this exercise, you will explore how robust the bike network is to the loss of its most important hubs. Start by identifying a small set of central stations (for example, using a centrality or hub measure from a previous exercise) and then simulate their failure by removing them and their incident edges from the graph. Compare the connectivity of the original and modified networks: examine changes in the size of the largest connected component, the number of isolated stations, and the distribution of component sizes. Use these comparisons to argue whether the system is resilient to the loss of key hubs or whether a few stations are critical for maintaining overall connectivity.

